# Supplementary Figure (Fig 3) 1 - Treatment design: induced-vs-sample confound + CRISPR induction

Sub-panels under `Fig2/supp_1/<panel>/`. **S1a** per-sample composition + the induced-vs-sample
confound that mandates pseudobulk; **S1b** per-lineage pseudotime QC. (A human PAS-distance panel
is not built: the human peak-to-AATAAA distances are not cached and would need a genome-level
recompute - flagged, analogous to mouse S1a.)


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

FIG = "Fig3/supp_1"
H5 = f"{DATA_ROOT}/3utr/organoids/knoblich/cellranger/aggregated_annotated_fix_filtered_normalized_dpt.h5ad"
adata = sc.read_h5ad(H5, backed="r")
obs = adata.obs.copy()
print("cells:", len(obs), "| samples:", obs["sample"].nunique())

In [ ]:
# S1a - per-sample cell counts coloured by induction (14-vs-8 confound).
# Cells are filtered to MATCH s1c: induced cells carrying Control2 / no assigned guide are
# removed, so the total equals the control+KO set shown in s1c (not the raw atlas count).
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch

CTRL_TEAL, KO_GOLD = "#298c8c", "#f1a226"  # teal = control/uninduced, gold = KO/induced
_gg = obs["gRNA"].astype(str)
_ii = obs["induced"].astype(str)
_keep_kd = ~((_ii == "True") & ((_gg == "nan") | (_gg == "Control2")))
obs_kd = obs[_keep_kd]
gsa = (
    obs_kd.groupby("sample")
    .agg(n=("induced", "size"), induced=("induced", lambda x: str(x.iloc[0])))
    .reset_index()
    .sort_values(["induced", "n"])
)
n_ind = int((gsa.induced == "True").sum())
n_unind = int((gsa.induced == "False").sum())
print(f"S1a cells after exclusion: {int(gsa.n.sum()):,} (matches s1c control+KO)")


def _build_s1a(figsize, mode):
    fig, ax = plt.subplots(figsize=figsize)
    cols = [KO_GOLD if i == "True" else CTRL_TEAL for i in gsa.induced]
    ax.bar(np.arange(len(gsa)), gsa.n, color=cols, edgecolor="none")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="y", labelsize=S.FONT_TICK)
    if mode == "full":
        ax.set_xticks(np.arange(len(gsa)))
        ax.set_xticklabels(gsa["sample"], rotation=90, fontsize=S.FONT_TICK - 1.5)
        ax.set_ylabel("cells per sample", fontsize=S.FONT_LABEL)
        ax.legend(
            handles=[
                Patch(facecolor=KO_GOLD, label=f"induced ({n_ind} samples)"),
                Patch(facecolor=CTRL_TEAL, label=f"uninduced ({n_unind} samples)"),
            ],
            fontsize=S.FONT_LEGEND,
            frameon=False,
            loc="upper left",
        )
        ax.set_title(
            "`induced` is constant within sample (14 vs 8) -> the perturbation contrast must be pseudobulk",
            fontsize=S.FONT_ANNOT - 0.5,
            pad=3,
        )
        fig.subplots_adjust(left=0.09, right=0.97, top=0.91, bottom=0.20)
    else:
        ax.set_xticks([])
        ax.set_xlabel("samples (sorted by induction)", fontsize=S.FONT_LABEL)
        ax.set_ylabel("cells/sample", fontsize=S.FONT_LABEL)
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
        ax.legend(
            handles=[
                Patch(facecolor=KO_GOLD, label=f"induced ({n_ind})"),
                Patch(facecolor=CTRL_TEAL, label=f"uninduced ({n_unind})"),
            ],
            fontsize=S.FONT_LEGEND,
            frameon=False,
            loc="upper left",
            handlelength=1.0,
            handletextpad=0.4,
            borderpad=0.1,
        )
        fig.subplots_adjust(
            left=0.13,
            right=0.98,
            top=0.96 if mode == "quarter" else 0.97,
            bottom=0.16 if mode == "quarter" else 0.24,
        )
    return fig


S.save_and_show(
    _build_s1a((S.FULL_W_PX / 300, 760 / 300), "full"),
    "fig_s1a_sample_design",
    figure=FIG,
)
S.save_and_show(
    _build_s1a((S.HALF_W_IN, S.QUARTER_H_IN), "quarter"),
    "fig_s1a_sample_design__quarter_half",
    figure=FIG,
)
S.save_and_show(
    _build_s1a((S.HALF_W_IN, S.EIGHTH_H_IN), "eighth"),
    "fig_s1a_sample_design__eighth_half",
    figure=FIG,
)

In [ ]:
# S1c - CRISPR induction on the atlas UMAP: controls vs confirmed 36-ASD-risk-guide cells.
# Induced cells carrying Control2 / no guide are removed (not confirmed KOs). Random order.
from matplotlib.lines import Line2D

CTRL_TEAL, KO_GOLD = "#298c8c", "#f1a226"
EMB = S.OUT_ROOT / "Fig2" / "fig2a_umap" / "_embedding_pcs20_nn30_md0.3_sp1.0_s0.npy"
um = np.load(EMB)
_g = obs["gRNA"].astype(str).values
_ind = obs["induced"].astype(str).values
klass = np.full(len(obs), "exclude", dtype=object)
klass[_ind == "False"] = "control"
klass[(_ind == "True") & (_g != "nan") & (_g != "Control2")] = "induced_targeting"
n_ctrl = int((klass == "control").sum())
n_kd = int((klass == "induced_targeting").sum())
n_excl = int((klass == "exclude").sum())
keep = klass != "exclude"
pal = {"control": CTRL_TEAL, "induced_targeting": KO_GOLD}


def _build_s1c(figsize, mode):
    rgba = S.palette_rgba(klass[keep], pal, alpha=0.5)
    fig, ax = plt.subplots(figsize=figsize)
    S.scatter_random_order(
        ax,
        um[keep, 0],
        um[keep, 1],
        rgba=rgba,
        s=1.0 if mode == "full" else 1.2,
        seed=0,
    )
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    if mode == "full":
        h = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=CTRL_TEAL,
                markersize=4,
                label=f"control / uninduced (n={n_ctrl:,})",
            ),
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=KO_GOLD,
                markersize=4,
                label=f"36-ASD-risk-guide (n={n_kd:,})",
            ),
        ]
        ax.legend(
            handles=h,
            loc="lower left",
            fontsize=S.FONT_LEGEND,
            frameon=False,
            handletextpad=0.2,
        )
        ax.set_title(
            f"Controls vs confirmed 36-ASD-risk-guide cells\n({n_excl:,} Control2/unassigned induced cells removed)",
            fontsize=S.FONT_ANNOT - 0.5,
            pad=3,
        )
        ax.set_xlabel("UMAP 1", fontsize=S.FONT_LABEL)
        ax.set_ylabel("UMAP 2", fontsize=S.FONT_LABEL)
        fig.subplots_adjust(left=0.05, right=0.96, top=0.86, bottom=0.07)
    else:
        ax.text(
            0.985,
            0.985,
            "Gene expression",
            transform=ax.transAxes,
            fontsize=S.FONT_LEGEND,
            color="0.4",
            va="top",
            ha="right",
            style="italic",
        )
        h = [
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=CTRL_TEAL,
                markersize=3.5,
                label=f"control (n={n_ctrl:,})",
            ),
            Line2D(
                [0],
                [0],
                marker="o",
                color="w",
                markerfacecolor=KO_GOLD,
                markersize=3.5,
                label=f"36-ASD-risk KO (n={n_kd:,})",
            ),
        ]
        ax.legend(
            handles=h,
            loc="lower left",
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            handletextpad=0.2,
            borderpad=0.0,
            labelspacing=0.2,
            bbox_to_anchor=(-0.02, -0.02),
        )
        fig.subplots_adjust(left=0.03, right=0.985, top=0.985, bottom=0.03)
    return fig


S.save_and_show(
    _build_s1c((S.HALF_W_PX / 300, 720 / 300), "full"),
    "fig_s1c_crispr_induction",
    figure=FIG,
)
S.save_and_show(
    _build_s1c((S.HALF_W_IN, S.QUARTER_H_IN), "quarter"),
    "fig_s1c_crispr_induction__quarter_half",
    figure=FIG,
)
S.save_and_show(
    _build_s1c((S.HALF_W_IN, S.EIGHTH_H_IN), "eighth"),
    "fig_s1c_crispr_induction__eighth_half",
    figure=FIG,
)

## S1c variant: CRISPR induction on the **APA** UMAP (per lineage)

The same control-vs-confirmed-36-ASD-risk-guide contrast as S1c, but on the **3'UTR-proportion
(APA) UMAP** (embedding from `merged_bw12_..._percentages_annotated_harmony.h5ad`, as in Fig 2b)
instead of the gene-expression atlas, and split by lineage (2 UMAPs). A fresh blue/red palette
(not used elsewhere: treatment = magenta/green, lineage = orange/purple). Random order; confirmed
KO = induced & a targeting guide (Control2 / unassigned removed). Saved
`fig_s1c_crispr_induction__apa_exc` and `__apa_inh`.


In [ ]:
# CRISPR induction on the APA UMAP, per lineage. teal/gold (control/KO). Base keeps the lineage
# title; the quarter/eighth variants carry a small "UTR length proportions" tag (top-right) so
# readers know this is APA, not gene expression.
from matplotlib.lines import Line2D

_APA_H5 = (
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/"
    "merged_bw12_above10cts_apa_named_percentages_annotated_harmony.h5ad"
)
_ada = sc.read_h5ad(_APA_H5, backed="r")
_um = np.asarray(_ada.obsm["X_umap"])
_lin = _ada.obs["lineage"].astype(str).values
_g = _ada.obs["gRNA"].astype(str).values
_i = _ada.obs["induced"].astype(str).values
_kl = np.full(len(_lin), "exclude", dtype=object)
_kl[_i == "False"] = "control"
_kl[(_i == "True") & (_g != "nan") & (_g != "Control2")] = "induced_targeting"
_PAL_APA = {"control": "#298c8c", "induced_targeting": "#f1a226"}


def _apa_umap(lineage_label, stem, figsize, mode):
    m = (_lin == lineage_label) & (_kl != "exclude")
    kk = _kl[m]
    nc = int((kk == "control").sum())
    nk = int((kk == "induced_targeting").sum())
    rgba = S.palette_rgba(kk, _PAL_APA, alpha=0.55)
    fig, ax = plt.subplots(figsize=figsize)
    S.scatter_random_order(ax, _um[m, 0], _um[m, 1], rgba=rgba, s=2.0, seed=0)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    key = lineage_label.split()[0].lower()
    if mode == "full":
        ax.set_title(
            lineage_label,
            fontsize=S.FONT_LABEL,
            color=S.paper_palette_lineage[key],
            pad=2,
        )
        ax.set_xlabel("APA UMAP 1", fontsize=S.FONT_LABEL)
        ax.set_ylabel("APA UMAP 2", fontsize=S.FONT_LABEL)
        ax.text(
            0.985,
            0.985,
            "UTR length proportions",
            transform=ax.transAxes,
            fontsize=S.FONT_LEGEND,
            color="0.4",
            va="top",
            ha="right",
            style="italic",
        )  # data-type tag on the base panels too
        lab_c, lab_k = (
            f"control / uninduced (n={nc:,})",
            f"36-ASD-risk-guide (n={nk:,})",
        )
        fig.subplots_adjust(left=0.07, right=0.98, top=0.90, bottom=0.10)
    else:
        ax.text(
            0.015,
            0.985,
            lineage_label.split()[0][:3].upper(),
            transform=ax.transAxes,
            fontsize=S.FONT_ANNOT,
            color=S.paper_palette_lineage[key],
            fontweight="bold",
            va="top",
            ha="left",
        )
        ax.text(
            0.985,
            0.985,
            "UTR length proportions",
            transform=ax.transAxes,
            fontsize=S.FONT_LEGEND,
            color="0.4",
            va="top",
            ha="right",
            style="italic",
        )
        lab_c, lab_k = f"control (n={nc:,})", f"36-ASD-risk KO (n={nk:,})"
        fig.subplots_adjust(left=0.03, right=0.985, top=0.985, bottom=0.03)
    h = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=_PAL_APA["control"],
            markersize=3.5,
            label=lab_c,
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=_PAL_APA["induced_targeting"],
            markersize=3.5,
            label=lab_k,
        ),
    ]
    ax.legend(
        handles=h,
        loc="lower left",
        fontsize=S.FONT_LEGEND - 0.5,
        frameon=False,
        handletextpad=0.2,
        borderpad=0.0,
        labelspacing=0.2,
        bbox_to_anchor=(-0.02, -0.02),
    )
    S.save_and_show(fig, f"fig_s1c_crispr_induction__apa_{stem}", figure=FIG)
    print(f"{lineage_label}: control={nc:,} | 36-ASD-risk-guide={nk:,}")


for _L, _s in [("Excitatory neurons", "exc"), ("Inhibitory neurons", "inh")]:
    _apa_umap(
        _L, _s, (S.HALF_W_PX / 300, S.QUARTER_H_PX / 300), "full"
    )  # base (palette-updated)
    _apa_umap(_L, f"{_s}_quarter_half", (S.HALF_W_IN, S.QUARTER_H_IN), "quarter")
    _apa_umap(_L, f"{_s}_eighth_half", (S.HALF_W_IN, S.EIGHTH_H_IN), "eighth")